<a href="https://colab.research.google.com/github/MMujtabaX/agentic-ai-journey/blob/main/02_rag_vector_search/faiss_vector_search.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 02 · Vector Search with FAISS + LangChain

The **retrieval** half of RAG: turn text into embeddings, store them in a **FAISS** index, and find the most relevant documents for a query, with metadata filtering, similarity scores and MMR retrieval. Then index real **Wikipedia** articles and search them.

Free to run: local Hugging Face embeddings, no API key needed.

In [1]:
!pip install -qU langchain-community langchain-huggingface faiss-cpu wikipedia

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 25.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 2.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


## 1. Embeddings
`all-mpnet-base-v2` maps any text to a 768-dimensional vector; similar meanings land close together.

In [2]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")
dim = len(embeddings.embed_query("hello world"))
print("Embedding dimension:", dim)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/11.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding dimension: 768


## 2. Create an empty FAISS vector store
`IndexFlatL2` does exact nearest-neighbour search by Euclidean distance.

In [3]:
import faiss
from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_community.vectorstores import FAISS

vector_store = FAISS(
    embedding_function=embeddings,
    index=faiss.IndexFlatL2(dim),
    docstore=InMemoryDocstore(),
    index_to_docstore_id={},
)

/tmp/ipykernel_1525/4188074737.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.docstore.in_memory import InMemoryDocstore


## 3. Add documents with metadata
Each document carries a `source` tag so searches can be filtered.

In [5]:
from uuid import uuid4

from langchain_core.documents import Document

document_1 = Document(
    page_content="I had chocolate chip pancakes and scrambled eggs for breakfast this morning.",
    metadata={"source": "tweet"},
)

document_2 = Document(
    page_content="The weather forecast for tomorrow is cloudy and overcast, with a high of 62 degrees.",
    metadata={"source": "news"},
)

document_3 = Document(
    page_content="Building an exciting new project with LangChain - come check it out!",
    metadata={"source": "tweet"},
)

document_4 = Document(
    page_content="Robbers broke into the city bank and stole $1 million in cash.",
    metadata={"source": "news"},
)

document_5 = Document(
    page_content="Wow! That was an amazing movie. I can't wait to see it again.",
    metadata={"source": "tweet"},
)

document_6 = Document(
    page_content="Is the new iPhone worth the price? Read this review to find out.",
    metadata={"source": "website"},
)

document_7 = Document(
    page_content="The top 10 soccer players in the world right now.",
    metadata={"source": "website"},
)

document_8 = Document(
    page_content="LangGraph is the best framework for building stateful, agentic applications!",
    metadata={"source": "tweet"},
)

document_9 = Document(
    page_content="The stock market is down 500 points today due to fears of a recession.",
    metadata={"source": "news"},
)

document_10 = Document(
    page_content="I have a bad feeling I am going to get deleted :(",
    metadata={"source": "tweet"},
)
document_11 = Document(
    page_content="Open AI comes up with a coding IDE called codex",
    metadata={"source": "X"},
)
# document_12 = Document(
#     page_content=docs,
#     metadata={"source": "wiki"},
# )
documents = [
    document_1,
    document_2,
    document_3,
    document_4,
    document_5,
    document_6,
    document_7,
    document_8,
    document_9,
    document_10,
    document_11
]
uuids = [str(uuid4()) for _ in range(len(documents))]

vector_store.add_documents(documents=documents, ids=uuids)

['cb5c0d18-d4d0-4aca-b648-e3815261c436',
 'c290c3f0-4464-4605-a426-c476357d4322',
 '0e2dbb6d-3bdc-49f5-8972-e5e2469b89ca',
 '72c81ca4-abf1-441a-b7e2-45b9cddd049d',
 '9604754e-5276-409c-bc0d-09c6a22aeb8d',
 'bc28bd8c-6ecf-4347-81aa-c02c2aa31b77',
 '2a2acd3a-9309-415a-8f94-4cc7a144d47f',
 'cb8cccd6-b9d9-40f7-b312-4ec6abc5531b',
 'b951f659-357a-48c0-af68-27b0e1352ca5',
 '5a965ebb-2e66-4af9-8dc5-fa4b0b60f923',
 'e8b8d7b2-7160-4d09-a0c1-453922310319']

In [6]:
# Delete an item from the vector store
vector_store.delete(ids=[uuids[-1]])
print("Documents in store:", vector_store.index.ntotal)

Documents in store: 10


## 4. Search

In [7]:
results = vector_store.similarity_search("LangChain provides abstractions to make working with LLMs easy",
                                         k=2, filter={"source": "tweet"})
for res in results:
    print(f"* {res.page_content} [{res.metadata}]")

* Building an exciting new project with LangChain - come check it out! [{'source': 'tweet'}]
* LangGraph is the best framework for building stateful, agentic applications! [{'source': 'tweet'}]


In [8]:
# L2 distance: lower = more similar
results = vector_store.similarity_search_with_score("Will it be hot tomorrow?", k=1, filter={"source": "news"})
for res, score in results:
    print(f"* [distance={score:.3f}] {res.page_content} [{res.metadata}]")

* [distance=0.809] The weather forecast for tomorrow is cloudy and overcast, with a high of 62 degrees. [{'source': 'news'}]


In [9]:
# MMR (maximal marginal relevance) balances relevance and diversity
retriever = vector_store.as_retriever(search_type="mmr", search_kwargs={"k": 1})
retriever.invoke("Stealing from the bank is a crime", filter={"source": "news"})

[Document(id='72c81ca4-abf1-441a-b7e2-45b9cddd049d', metadata={'source': 'news'}, page_content='Robbers broke into the city bank and stole $1 million in cash.')]

## 5. Index real Wikipedia articles
Fetch articles with `WikipediaRetriever`, split them into chunks, add them to the same index, and search.

In [10]:
from langchain_community.retrievers import WikipediaRetriever
from langchain_text_splitters import RecursiveCharacterTextSplitter

wiki = WikipediaRetriever(top_k_results=3, doc_content_chars_max=4000)
wiki_docs = wiki.invoke("Artificial intelligence") + wiki.invoke("OpenAI")
print("Wikipedia pages fetched:", [d.metadata["title"] for d in wiki_docs])

chunks = RecursiveCharacterTextSplitter(chunk_size=600, chunk_overlap=80).split_documents(wiki_docs)
for c in chunks:
    c.metadata = {"source": "wikipedia", "title": c.metadata.get("title", "")}
vector_store.add_documents(chunks)
print(f"Added {len(chunks)} Wikipedia chunks → store now holds {vector_store.index.ntotal} vectors")

Wikipedia pages fetched: ['Artificial intelligence', 'Artificial general intelligence', 'A.I. Artificial Intelligence', 'OpenAI', 'OpenAI–HuggingFace incident', 'OpenAI Codex']
Added 65 Wikipedia chunks → store now holds 75 vectors


In [11]:
for query in ["Who founded OpenAI?", "What is machine learning?", "What are the risks of AI?"]:
    print(f"\n🔎 {query}")
    for doc, score in vector_store.similarity_search_with_score(query, k=2, filter={"source": "wikipedia"}):
        print(f"  [{score:.3f}] ({doc.metadata['title']}) {doc.page_content[:160].strip()}...")


🔎 Who founded OpenAI?
  [0.648] (OpenAI) In December 2015, OpenAI was founded as the nonprofit organization OpenAI, Inc., by Elon Musk, Sam Altman, Ilya Sutskever, Greg Brockman, Trevor Blackwell, Vick...
  [0.668] (OpenAI) Former OpenAI personnel have founded competing AI companies including Anthropic, Safe Superintelligence Inc., SpaceXAI, and Thinking Machines Lab....

🔎 What is machine learning?
  [0.751] (Artificial intelligence) Artificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoni...
  [1.067] (Artificial general intelligence) Artificial general intelligence (AGI) is a hypothetical type of artificial intelligence that matches or surpasses human capabilities across virtually all cognit...

🔎 What are the risks of AI?
  [0.688] (Artificial intelligence) AI boom coincided with advances in generative AI, which became widespread and allowed for the creation and modification of 